In [14]:
import pandas as pd
import numpy as np

rayyan_clean = pd.read_csv('/data1/qianc/EMCL/datasets/rayyan/clean.csv', na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')
rayyan_dirty = pd.read_csv('/data1/qianc/EMCL/datasets/rayyan/dirty.csv', na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')

rayyan_clean.head()
rayyan_dirty.head()
error_df = rayyan_clean != rayyan_dirty

for i in range(rayyan_clean.shape[0]):
    for j in range(rayyan_clean.shape[1]):
        if error_df.iloc[i, j]:
            print(rayyan_clean.iloc[i, j], rayyan_dirty.iloc[i, j], f"index: {i}, {j}")
            


-1  index: 1, 6
-1  index: 1, 7
2/15/04 4/2/15 index: 1, 8
1/6/12 12/1/06 index: 2, 8
1/13/01 1/1/13 index: 3, 8
1/15/10 10/1/15 index: 4, 8
1/9/01 1/1/09 index: 5, 8
1/1/07 7/1/01 index: 7, 8
{"St_ber F","Mutz N","Wrigge H","Putensen C","Zinserling J","Zech S","Von Spiegel T"} {"St��_ber F","Mutz N","Wrigge H","Putensen C","Zinserling J","Zech S","Von Spiegel T"} index: 7, 10
1/11/01 1/1/11 index: 8, 8
1/6/01 1/1/06 index: 9, 8
1/9/01 1/1/09 index: 10, 8
1/15/01 1/1/15 index: 11, 8
1/13/01 1/1/13 index: 12, 8
1/11/01 1/1/11 index: 13, 8
1/11/01 1/1/11 index: 14, 8
-1  index: 16, 7
1/12/01 1/1/12 index: 16, 8
1/5/01 1/1/05 index: 17, 8
1/7/06 6/1/07 index: 19, 8
1/13/12 12/1/13 index: 20, 8
1/11/01 1/1/11 index: 21, 8
1/14/12 12/1/14 index: 22, 8
1/12/01 1/1/12 index: 24, 8
1/14/03 3/1/14 index: 25, 8
1/8/12 12/1/08 index: 26, 8
1/13/01 1/1/13 index: 28, 8
1/9/01 1/1/09 index: 29, 8
1/8/12 12/1/08 index: 30, 8
1/7/09 9/1/07 index: 31, 8
-1  index: 33, 7
1/15/02 2/1/15 index: 34, 8
1/8/

In [1]:
# 使用 sklearn 的 f1_score 实现数据修复评估
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd
import numpy as np

def calculate_f1_with_sklearn(df_clean, df_repaired, df_dirty):
    """
    使用 sklearn 的 f1_score 计算数据修复的 F1 分数
    
    参数:
        df_clean: 干净数据（ground truth）
        df_repaired: 修复后的数据
        df_dirty: 原始脏数据
    
    返回:
        dict: 包含 precision, recall, f1_score 的字典
    """
    # 将 DataFrame 转换为布尔矩阵：True 表示正确，False 表示错误
    # 对于修复后的数据
    repaired_correct = (df_clean == df_repaired).values.flatten()
    
    # 对于脏数据
    dirty_correct = (df_clean == df_dirty).values.flatten()
    
    # 将布尔值转换为 0/1：1 表示正确，0 表示错误
    y_true = dirty_correct.astype(int)  # 原始状态：哪些单元格是正确的
    y_pred = repaired_correct.astype(int)  # 修复后的状态：哪些单元格是正确的
    
    # 计算指标
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

def custom_f1_score(df_clean, df_repaired, df_dirty):
    """
    参考 utils.py 中的逻辑实现的 F1 分数
    
    参数:
        df_clean: 干净数据
        df_repaired: 修复后的数据  
        df_dirty: 原始脏数据
    """
    new_errors = (df_clean != df_repaired)
    old_errors = (df_clean != df_dirty)
    new_errors_count = int(new_errors.values.sum())
    old_errors_count = int(old_errors.values.sum())
    
    if old_errors_count == 0:
        return 1.0  # 如果原始数据没有错误，返回 1
    
    recall = (old_errors_count - new_errors_count) / old_errors_count
    f1 = 2 * recall / (1 + recall) if (1 + recall) != 0 else 0
    
    return f1

print("F1 Score 计算方法已定义完成！")


F1 Score 计算方法已定义完成！


In [2]:
# 在 beers 数据集上测试 F1 Score 方法
experiment = "beers"

# 加载数据
beers_clean = pd.read_csv(f'/data1/qianc/EMCL/datasets/{experiment}/clean.csv', 
                          na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')
beers_dirty = pd.read_csv(f'/data1/qianc/EMCL/datasets/{experiment}/dirty.csv', 
                          na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')

# 检查是否有修复后的数据，如果没有则使用 dirty 作为示例
try:
    beers_repaired = pd.read_csv(f'/data1/qianc/result/{experiment}_repaired_original.csv', 
                                  na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')
    print(f"已加载 beers 修复数据")
except:
    beers_repaired = beers_dirty.copy()
    print(f"未找到修复数据，使用 dirty 数据作为示例")

print(f"\n数据集形状:")
print(f"Clean: {beers_clean.shape}")
print(f"Dirty: {beers_dirty.shape}")
print(f"Repaired: {beers_repaired.shape}")

# 计算错误统计
error_count = (beers_clean != beers_dirty).sum().sum()
total_cells = beers_clean.shape[0] * beers_clean.shape[1]
print(f"\n错误统计:")
print(f"总单元格数: {total_cells}")
print(f"错误单元格数: {error_count}")
print(f"错误率: {error_count/total_cells*100:.2f}%")


已加载 beers 修复数据

数据集形状:
Clean: (2410, 10)
Dirty: (2410, 10)
Repaired: (2410, 10)

错误统计:
总单元格数: 24100
错误单元格数: 5012
错误率: 20.80%


In [3]:
# 计算和比较 F1 Score

print("=" * 60)
print("F1 Score 计算结果 - Beers 数据集")
print("=" * 60)

# 方法1：使用 sklearn 的 f1_score
sklearn_results = calculate_f1_with_sklearn(beers_clean, beers_repaired, beers_dirty)
print("\n方法1：使用 sklearn.metrics.f1_score")
print(f"  Precision: {sklearn_results['precision']:.4f}")
print(f"  Recall:    {sklearn_results['recall']:.4f}")
print(f"  F1 Score:  {sklearn_results['f1_score']:.4f}")

# 方法2：使用 utils.py 中的自定义方法
custom_f1 = custom_f1_score(beers_clean, beers_repaired, beers_dirty)
print(f"\n方法2：自定义 F1 Score（参考 utils.py）")
print(f"  F1 Score:  {custom_f1:.4f}")

# 计算修复效果
new_errors = (beers_clean != beers_repaired).sum().sum()
old_errors = (beers_clean != beers_dirty).sum().sum()
fixed_errors = old_errors - new_errors

print(f"\n修复统计:")
print(f"  原始错误数: {old_errors}")
print(f"  修复后错误数: {new_errors}")
print(f"  修复的错误数: {fixed_errors}")
print(f"  修复率: {(fixed_errors/old_errors*100) if old_errors > 0 else 0:.2f}%")

print("\n" + "=" * 60)


F1 Score 计算结果 - Beers 数据集

方法1：使用 sklearn.metrics.f1_score
  Precision: 0.9970
  Recall:    1.0000
  F1 Score:  0.9985

方法2：自定义 F1 Score（参考 utils.py）
  F1 Score:  0.0225

修复统计:
  原始错误数: 5012
  修复后错误数: 4955
  修复的错误数: 57
  修复率: 1.14%



In [5]:
# 正确的 F1 Score 计算方法
def correct_f1_score(df_clean, df_repaired, df_dirty):
    """
    正确计算数据修复的 F1 分数
    
    将修复问题看作：哪些错误单元格被正确修复了
    - TP (True Positive): 原本错误，被正确修复
    - FP (False Positive): 原本正确，被错误修改
    - FN (False Negative): 原本错误，没有修复或修复错误
    """
    old_errors = (df_clean != df_dirty)  # 原始错误位置
    new_errors = (df_clean != df_repaired)  # 修复后仍然错误的位置
    
    # TP: 原本错误，现在正确了（被成功修复）
    TP = (old_errors & ~new_errors).sum().sum()
    
    # FP: 原本正确，现在错误了（引入了新错误）
    FP = (~old_errors & new_errors).sum().sum()
    
    # FN: 原本错误，现在仍然错误（未修复或修复失败）
    FN = (old_errors & new_errors).sum().sum()
    
    # 计算指标
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'TP': TP,
        'FP': FP, 
        'FN': FN,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

# 在 beers 数据集上测试正确的方法
print("=" * 60)
print("正确的 F1 Score 计算（基于错误修复）")
print("=" * 60)

correct_results = correct_f1_score(beers_clean, beers_repaired, beers_dirty)

print(f"\n混淆矩阵:")
print(f"  TP (成功修复的错误): {correct_results['TP']}")
print(f"  FP (引入的新错误):   {correct_results['FP']}")
print(f"  FN (未修复的错误):   {correct_results['FN']}")

print(f"\n评估指标:")
print(f"  Precision: {correct_results['precision']:.4f}  (修复的准确性)")
print(f"  Recall:    {correct_results['recall']:.4f}  (修复的覆盖率)")
print(f"  F1 Score:  {correct_results['f1_score']:.4f}")

print(f"\n解释:")
print(f"  - 原始错误数: {correct_results['TP'] + correct_results['FN']}")
print(f"  - 成功修复数: {correct_results['TP']}")
print(f"  - 修复率: {correct_results['recall']*100:.2f}%")
print(f"  - 修复准确率: {correct_results['precision']*100:.2f}%")
print("=" * 60)


正确的 F1 Score 计算（基于错误修复）

混淆矩阵:
  TP (成功修复的错误): 57
  FP (引入的新错误):   0
  FN (未修复的错误):   4955

评估指标:
  Precision: 1.0000  (修复的准确性)
  Recall:    0.0114  (修复的覆盖率)
  F1 Score:  0.0225

解释:
  - 原始错误数: 5012
  - 成功修复数: 57
  - 修复率: 1.14%
  - 修复准确率: 100.00%


In [6]:
# 分析 utils.py 中方法的问题
print("=" * 60)
print("utils.py 方法的问题分析")
print("=" * 60)

# utils.py 的方法
old_errors_count = (beers_clean != beers_dirty).sum().sum()
new_errors_count = (beers_clean != beers_repaired).sum().sum()

recall_utils = (old_errors_count - new_errors_count) / old_errors_count
f1_utils = 2 * recall_utils / (1 + recall_utils)

print(f"\nutils.py 的计算逻辑:")
print(f"  old_errors_count = {old_errors_count}")
print(f"  new_errors_count = {new_errors_count}")
print(f"  recall = (old - new) / old = ({old_errors_count} - {new_errors_count}) / {old_errors_count} = {recall_utils:.4f}")
print(f"  f1 = 2*recall/(1+recall) = {f1_utils:.4f}")

print(f"\n❌ 问题:")
print(f"  1. 这个方法假设 precision = 1（没有引入新错误）")
print(f"  2. 公式 f1 = 2*recall/(1+recall) 等价于假设 precision=1 的标准 F1 公式")
print(f"  3. 实际上:")
print(f"     - 成功修复: {correct_results['TP']} 个错误")
print(f"     - 引入新错误: {correct_results['FP']} 个")
print(f"     - 未修复: {correct_results['FN']} 个错误")

print(f"\n✅ 正确的 F1 计算应该同时考虑:")
print(f"  - Precision = TP/(TP+FP) = {correct_results['TP']}/({correct_results['TP']}+{correct_results['FP']}) = {correct_results['precision']:.4f}")
print(f"  - Recall = TP/(TP+FN) = {correct_results['TP']}/({correct_results['TP']}+{correct_results['FN']}) = {correct_results['recall']:.4f}")  
print(f"  - F1 = 2*P*R/(P+R) = {correct_results['f1_score']:.4f}")

print("=" * 60)


utils.py 方法的问题分析

utils.py 的计算逻辑:
  old_errors_count = 5012
  new_errors_count = 4955
  recall = (old - new) / old = (5012 - 4955) / 5012 = 0.0114
  f1 = 2*recall/(1+recall) = 0.0225

❌ 问题:
  1. 这个方法假设 precision = 1（没有引入新错误）
  2. 公式 f1 = 2*recall/(1+recall) 等价于假设 precision=1 的标准 F1 公式
  3. 实际上:
     - 成功修复: 57 个错误
     - 引入新错误: 0 个
     - 未修复: 4955 个错误

✅ 正确的 F1 计算应该同时考虑:
  - Precision = TP/(TP+FP) = 57/(57+0) = 1.0000
  - Recall = TP/(TP+FN) = 57/(57+4955) = 0.0114
  - F1 = 2*P*R/(P+R) = 0.0225


In [12]:
a = rayyan_clean.iloc[1, 9]
b = rayyan_dirty.iloc[1, 9]
print(a)
print(b)
print(type(a))
print(type(b))
print(a == b)
print(np.isnan(a))
print(np.isnan(b))
print(np.isnan(a) == np.isnan(b))


nan
nan
<class 'float'>
<class 'float'>
False
True
True
True


In [7]:
# 简洁版本：按列统计差异
def compare_by_column(df_clean, df_repaired, df_dirty):
    """
    按列统计修复情况
    """
    print("=" * 100)
    print("按列统计修复情况")
    print("=" * 100)
    
    for col_idx, col_name in enumerate(df_clean.columns):
        # 统计该列的情况
        col_clean = df_clean.iloc[:, col_idx]
        col_repaired = df_repaired.iloc[:, col_idx]
        col_dirty = df_dirty.iloc[:, col_idx]
        
        # 原始错误
        original_errors = (col_clean != col_dirty).sum()
        # 当前错误
        current_errors = (col_clean != col_repaired).sum()
        # 成功修复
        fixed = original_errors - current_errors
        # 引入新错误
        new_errors = ((col_clean == col_dirty) & (col_clean != col_repaired)).sum()
        
        if original_errors > 0 or current_errors > 0:
            print(f"\n列 '{col_name}' (索引 {col_idx}):")
            print(f"  原始错误: {original_errors}")
            print(f"  成功修复: {fixed}")
            print(f"  仍然错误: {current_errors}")
            print(f"  引入新错误: {new_errors}")
            print(f"  修复率: {(fixed/original_errors*100) if original_errors > 0 else 0:.1f}%")

# 按列统计
compare_by_column(beers_clean, beers_repaired, beers_dirty)


按列统计修复情况

列 'ounces' (索引 3):
  原始错误: 2410
  成功修复: 0
  仍然错误: 2410
  引入新错误: 0
  修复率: 0.0%

列 'abv' (索引 4):
  原始错误: 2348
  成功修复: 0
  仍然错误: 2348
  引入新错误: 0
  修复率: 0.0%

列 'city' (索引 8):
  原始错误: 127
  成功修复: 17
  仍然错误: 110
  引入新错误: 0
  修复率: 13.4%

列 'state' (索引 9):
  原始错误: 127
  成功修复: 40
  仍然错误: 87
  引入新错误: 0
  修复率: 31.5%


In [10]:
count = 0
for row in range(beers_repaired.shape[0]):
        for col in range(beers_repaired.shape[1]):
            if beers_repaired.iloc[row, col] != beers_clean.iloc[row, col]:
                print(f"clean: {beers_clean.iloc[row, col]}, repaired: {beers_repaired.iloc[row, col]}, dirty: {beers_dirty.iloc[row, col]}")
                print(f"type: {type(beers_clean.iloc[row, col])}, {type(beers_repaired.iloc[row, col])}, {type(beers_dirty.iloc[row, col])}")
                count += 1
        if count > 20:
            break


clean: 12.0, repaired: 12.0, dirty: 12.0 oz
type: <class 'numpy.float64'>, <class 'str'>, <class 'str'>
clean: 0.05, repaired: 0.05, dirty: 0.05
type: <class 'float'>, <class 'str'>, <class 'str'>
clean: 12.0, repaired: 12.0, dirty: 12.0 oz.
type: <class 'numpy.float64'>, <class 'str'>, <class 'str'>
clean: 0.066, repaired: 0.066, dirty: 0.066
type: <class 'float'>, <class 'str'>, <class 'str'>
clean: 12.0, repaired: 12.0, dirty: 12.0 ounce
type: <class 'numpy.float64'>, <class 'str'>, <class 'str'>
clean: 0.071, repaired: 0.071, dirty: 0.071
type: <class 'float'>, <class 'str'>, <class 'str'>
clean: 12.0, repaired: 12.0, dirty: 12.0 oz
type: <class 'numpy.float64'>, <class 'str'>, <class 'str'>
clean: 0.09, repaired: 0.065, dirty: 0.09%
type: <class 'float'>, <class 'str'>, <class 'str'>
clean: 12.0, repaired: 12.0, dirty: 12.0 OZ.
type: <class 'numpy.float64'>, <class 'str'>, <class 'str'>
clean: 0.075, repaired: 0.075, dirty: 0.075
type: <class 'float'>, <class 'str'>, <class 'str'>

In [ ]:
import numpy as np
import pandas as pd


def all_wrong_corrector(clean_df, dirty_df, error_df, prop=0.2):
    """
    used to correct columns that are all wrong
    """
    col_sum = error_df.sum(axis=0)
    for col in col_sum.index:
        # too many error, do correction
        if col_sum[col] > (1-prop) * error_df.shape[0]:
            print(col_sum[col], (1-prop) * error_df.shape[0])
            correct_sample = np.random.choice(error_df.shape[0], int(prop * error_df.shape[0]), replace=False)
            break
            dirty_df.loc[correct_sample, col] = clean_df.loc[correct_sample, col]
        else:
            continue

    error_df = clean_df.ne(dirty_df)
    return clean_df, dirty_df, error_df


tax = pd.read_csv('/data1/qianc/EMCL/datasets/tax_200k/clean.csv', na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')
tax_dirty = pd.read_csv('/data1/qianc/EMCL/datasets/tax_200k/dirty.csv', na_values=["nan", "NaN", "N/A", "None", "null"]).replace(np.nan, '')

error = tax.ne(tax_dirty)
a, b, c = all_wrong_corrector(tax, tax_dirty, error)
print(a.shape, b.shape, c.shape)




(17923, 15) (17913, 15) (17923, 15)


In [9]:
tax.ne(tax_dirty)

,f_name,l_name,gender,area_code,phone,city,state,zip,marital_status,has_child,salary,rate,single_exemp,married_exemp,child_exemp
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17918,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
17919,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
17920,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
17921,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
